In [3]:
import scanpy as sc
import numpy as np
import pandas as pd
import json

In [ ]:
#Load data and define query donors (same seed as main experiment)
adata = sc.read_h5ad('AIDA.h5ad', backed='r')

sg_chinese_donors = adata.obs[
    adata.obs['self_reported_ethnicity'] == 'Singaporean Chinese'
]['donor_id'].unique()

np.random.seed(42)
query_donors = np.random.choice(sg_chinese_donors, size=20, replace=False)
print(f"Query donors: {len(query_donors)}")

Query donors: 20


In [ ]:
#Check annotation quality of NK cells in query
query_mask = adata.obs['donor_id'].isin(query_donors)
nk_cells = adata.obs[query_mask & (adata.obs['cell_type'] == 'natural killer cell')]

print(f"Total NK cells in query: {len(nk_cells)}")
print()
print(nk_cells[['Annotation_Level1', 'Annotation_Level2',
                'Annotation_Level3', 'Annotation_Level4']].value_counts().head(20))

Total NK cells in query: 288

Annotation_Level1  Annotation_Level2  Annotation_Level3  Annotation_Level4   
NK                 NK                 NK                 flagged_NK_low_exp      174
                                                         flagged_platelet_sum    114
                                                         B_unknown                 0
                                                         CD4+_T_cm                 0
                                                         CD4+_T_cm_IFNhi           0
                                                         CD4+_T_cyt                0
                                                         CD4+_T_em                 0
                                                         CD4+_T_memory             0
                                                         CD4+_T_naive              0
                                                         CD4+_T_naive_IFNhi        0
                                          

In [ ]:
#Precision vs recall breakdown across proportions
with open('./output/all_reports.json', 'r') as f:
    all_reports = json.load(f)

rows = []
for entry in all_reports:
    if 'natural killer cell' in entry['report']:
        r = entry['report']['natural killer cell']
        rows.append({
            'proportion': entry['proportion'],
            'repeat': entry['repeat'],
            'precision': r['precision'],
            'recall': r['recall'],
            'f1': r['f1-score']
        })

df_nk = pd.DataFrame(rows)
summary = df_nk.groupby('proportion')[['precision', 'recall', 'f1']].mean().round(3)
print(summary)

            precision  recall     f1
proportion                          
0.0             0.704   0.527  0.601
0.1             0.719   0.577  0.638
0.2             0.703   0.587  0.638
0.3             0.732   0.581  0.647
0.4             0.725   0.612  0.663
0.5             0.743   0.623  0.677
0.6             0.718   0.610  0.659
0.7             0.711   0.598  0.649
0.8             0.721   0.606  0.658
0.9             0.723   0.601  0.656
1.0             0.745   0.616  0.675


In [ ]:
#Precision vs recall breakdown across proportions
with open('all_reports.json', 'r') as f:
    all_reports = json.load(f)

rows = []
for entry in all_reports:
    if 'natural killer cell' in entry['report']:
        r = entry['report']['natural killer cell']
        rows.append({
            'proportion': entry['proportion'],
            'repeat': entry['repeat'],
            'precision': r['precision'],
            'recall': r['recall'],
            'f1': r['f1-score']
        })

df_nk = pd.DataFrame(rows)
summary = df_nk.groupby('proportion')[['precision', 'recall', 'f1']].mean().round(3)
print(summary)

            precision  recall     f1
proportion                          
0.0             0.704   0.527  0.601
0.1             0.719   0.577  0.638
0.2             0.703   0.587  0.638
0.3             0.732   0.581  0.647
0.4             0.725   0.612  0.663
0.5             0.743   0.623  0.677
0.6             0.718   0.610  0.659
0.7             0.711   0.598  0.649
0.8             0.721   0.606  0.658
0.9             0.723   0.601  0.656
1.0             0.745   0.616  0.675


In [ ]:
# Replicate consistency: F1 difference between 100% and 0% SG_Chinese
pivot = df_nk.pivot(index='repeat', columns='proportion', values='f1')
pivot['diff_100_vs_0'] = pivot[1.0] - pivot[0.0]

print(pivot[['diff_100_vs_0']].describe().round(3))
print()
print(f"Replicates where 100% > 0%: {(pivot['diff_100_vs_0'] > 0).sum()} / {len(pivot)}")

proportion  diff_100_vs_0
count              33.000
mean                0.075
std                 0.053
min                -0.028
25%                 0.033
50%                 0.065
75%                 0.134
max                 0.157

Replicates where 100% > 0%: 32 / 36


In [ ]:
# Cell 6: Confirm the same flagged-cell pattern in other ethnicities' NK cells
other_ethnicities = ['Japanese', 'Korean', 'Thai', 'Indian',
                     'Singaporean Malay', 'Singaporean Indian']
other_mask = adata.obs['self_reported_ethnicity'].isin(other_ethnicities)
other_nk = adata.obs[other_mask & (adata.obs['cell_type'] == 'natural killer cell')]

print(f"Total NK cells in other ethnic groups: {len(other_nk)}")
print()
print(other_nk['Annotation_Level4'].value_counts().head(10))

Total NK cells in other ethnic groups: 21121

Annotation_Level4
NK_unknown              9199
flagged_platelet_sum    8553
flagged_NK_low_exp      3369
B_unknown                  0
CD4+_T_cm                  0
CD4+_T_cm_IFNhi            0
CD4+_T_cyt                 0
CD4+_T_em                  0
CD4+_T_memory              0
CD4+_T_naive               0
Name: count, dtype: int64


In [ ]:
# Cell 7: Check annotation quality of double negative T regulatory cell
dntreg = adata.obs[query_mask & (adata.obs['cell_type'] == 'double negative T regulatory cell')]
print(f"Total double negative T regulatory cells in query: {len(dntreg)}")
print()
print(dntreg['Annotation_Level4'].value_counts().head(10))

Total double negative T regulatory cells in query: 4

Annotation_Level4
dnT                    4
B_unknown              0
CD4+_T_cm              0
CD4+_T_cm_IFNhi        0
CD4+_T_cyt             0
CD4+_T_em              0
CD4+_T_memory          0
CD4+_T_naive           0
CD4+_T_naive_IFNhi     0
CD4+_T_naive_SOX4hi    0
Name: count, dtype: int64


In [ ]:
# Cell 8: Check annotation quality of innate lymphoid cell
ilc = adata.obs[query_mask & (adata.obs['cell_type'] == 'innate lymphoid cell')]
print(f"Total innate lymphoid cells in query: {len(ilc)}")
print()
print(ilc['Annotation_Level4'].value_counts().head(10))

Total innate lymphoid cells in query: 16

Annotation_Level4
ILC_XCL1hi            14
ILC_SCART1hi           2
B_unknown              0
CD4+_T_cm              0
CD4+_T_cm_IFNhi        0
CD4+_T_cyt             0
CD4+_T_em              0
CD4+_T_memory          0
CD4+_T_naive           0
CD4+_T_naive_IFNhi     0
Name: count, dtype: int64
